**MCQ Generator with Gemini**

In [ ]:
!pip install -q -U langchain-google-genai

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

**1. Import the dependencies...**

load the libraries for PDF reading (PyPDF2), data tables (pandas), prompts and JSON parsing (Langchain), and Colab secrets (userdata).

In [ ]:
import os
import pandas as pd
from google.colab import userdata
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from PyPDF2 import PdfReader

**2. Load API Key**

Read the Gemini key from Colab Secrets (keys icon in the left sidebar) and stores it as an environment variable.
The key is never written in the code, so it stays out of GitHub.

In [ ]:
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

**2. Check Available models**

Asks the Gemini API which models this key can use.
Model names change over time, so run this if you get a "404 NOT_FOUND" error, then pick a current 'flash' model.

In [ ]:
from google import genai

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
names = [m.name.replace("models/", "") for m in client.models.list()
  if "gemini" in m.name]
for n in names:
  print(n)

**4. Model with fallbacks**

Create one Gemini model per name in this list.
The first model is used by default. If it is busy (503 error) or unavailable, Langchain automatically tries the next one.

-'temperature=0.5': balanced between safe and creative answers

-'max_retries=3':retries each model before moving on

In [ ]:
names = ["gemini-3.5-flash", "gemini-3.5-flash-lite", "gemini-3.1-flash-lite"]

models = [
    ChatGoogleGenerativeAI(
        model = n,
        temperature=0.5,
        max_retries=3,
        google_api_key=userdata.get("GEMINI_API_KEY"),
    )
    for n in names
]

llm = models[0].with_fallbacks(models[1:])

**5. Prompt and chain**

The template tells the model to write MCQs using only the given text.
Placeholders '{number}', '{subject}', '{tone}' and '{text}' are filled in at run time. The chain 'prompt|llm|jsonOutputParser()' sends the prompt to the model and convert its JSON reply into Python dictionary.

In [ ]:
template = """
    Your are an expert MCQ Maker. Using ONLY the text below, create {number} multiple choice questions for {subject} students in a {tone} tone.
    Each question needs 4 options (a, b, c, d) with exactly one correct answer. Do not repeat questions. Return ONLY valid JSON in this format:
    {{"1": {{"mcq": "question text", "options": {{"a": "...", "b": "...", "c": "...", "d": "..."}}, "correct": "a"}}}}

    Text:{text}
"""
prompt = PromptTemplate.from_template(template)

chain = prompt | llm | JsonOutputParser()

**6. Helper Functions**

-'read_pdf(path)': extract the text from every page of a PDF.

-'generate_mcqs(...)': runs the chain and turns the result into a pandas table with columns MCQ, option a to d, and correct.

In [ ]:
def read_pdf(path):
  reader = PdfReader(path)
  return "".join(page.extract_text() or "" for page in reader.pages)

def generate_mcqs(text, number=5, subject="general", tone="simple"):
  result = chain.invoke({
      "text":text,
      "number":number,
      "subject":subject,
      "tone":tone,
  })
  rows = []
  for item in result.values():
    rows.append({
        "MCQ":item["mcq"],
        "option a":item["options"]["a"],
        "option b":item["options"]["b"],
        "option c":item["options"]["c"],
        "option d":item["options"]["d"],
        "correct":item["correct"],
    })
  return pd.DataFrame(rows)

**7. Generate and Save**

Reads the uploaded PDF, uses the first 6000 characters, generate 5 questions, saves them to 'mcqs.csv', and shows the table.

Change 'number', 'subject' and 'tone' to customise the output.


In [ ]:
text = read_pdf("/content/Lab-1-introduction-to-biology-Autosaved.pdf")
df = generate_mcqs(text[:6000], number=5, subject="biology", tone="simple")
df.to_csv("mcqs.csv", index=False)
df

**8. Debug check**

Prints the raw model output and how many questions it returned.

Use this to tell whether a problem comes from the model or from the table-building function.

In [ ]:
result = chain.invoke({"text":text[:6000], "number":5,"subject":"biology", "tone":"simple"})
print(len(result))
print(result)